In [24]:
# 1. Импорты
# Причина здесь для warnings — в коде LightAutoML, который использует устаревшую конкатенацию pandas, мне не удалось их подавить, так как они исходят не из моего кода, а изнутри LightAutoML, как пишут источники в Интернет
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.tasks import Task

In [25]:
# 2. Загрузка данных
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Объединение для анализа (без SalePrice в тесте)
combined_df = pd.concat([train_df.drop('SalePrice', axis=1), test_df], ignore_index=True)



# Проверка структуры
print("Размер train:", train_df.shape)
print("Размер test:", test_df.shape)
print("Объединенный размер:", combined_df.shape)
print("\nТипы данных в train:")
print(train_df.dtypes.value_counts())
print("\nПропуски в train:")
print(train_df.isnull().sum().sort_values(ascending=False).head(10))
print("\nПропуски в test:")
print(test_df.isnull().sum().sort_values(ascending=False).head(10))

# Разделение обратно
train_target = train_df['SalePrice']

Размер train: (1460, 81)
Размер test: (1459, 80)
Объединенный размер: (2919, 80)

Типы данных в train:
object     43
int64      35
float64     3
Name: count, dtype: int64

Пропуски в train:
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
dtype: int64

Пропуски в test:
PoolQC          1456
MiscFeature     1408
Alley           1352
Fence           1169
MasVnrType       894
FireplaceQu      730
LotFrontage      227
GarageYrBlt       78
GarageCond        78
GarageFinish      78
dtype: int64


In [26]:
# 3. Определение типов колонок для будущих ролей (LightAutoML сам определит, но для кастомных ролей)
numeric_cols = combined_df.select_dtypes(include=[np.number]).columns
categorical_cols = combined_df.select_dtypes(include=['object']).columns

# Формирование новых фичей
combined_df['OverallScore'] = combined_df['OverallQual'] * combined_df['OverallCond']  # Объединение качества и состояния дома (пример из задания)
combined_df['TotalSF'] = combined_df['TotalBsmtSF'] + combined_df['1stFlrSF'] + combined_df['2ndFlrSF']  # Общая жилая площадь
combined_df['Age'] = combined_df['YrSold'] - combined_df['YearBuilt']  # Возраст дома
combined_df['RemodAge'] = combined_df['YrSold'] - combined_df['YearRemodAdd']  # Возраст после ремонта
combined_df['TotalBath'] = combined_df['FullBath'] + 0.5 * combined_df['HalfBath'] + combined_df['BsmtFullBath'] + 0.5 * combined_df['BsmtHalfBath']  # Общее число ванных
combined_df['PorchSF'] = combined_df['OpenPorchSF'] + combined_df['EnclosedPorch'] + combined_df['3SsnPorch'] + combined_df['ScreenPorch']  # Общая площадь крыльца
combined_df['GarageScore'] = combined_df['GarageCars'] * combined_df['GarageArea']  # Оценка гаража


# Разделение обратно (без импьютации, LightAutoML сделает это)
train_features = combined_df.iloc[:len(train_df)].copy()
test_features = combined_df.iloc[len(train_df):].copy()

# Добавление целевой переменной к train
train_features['SalePrice'] = train_target

In [27]:
# 4. Настройка задачи
task = Task('reg', metric='mae')

# Базовый AutoML
automl = TabularAutoML(
    task=task,
    timeout=600,  # 10 минут
    cpu_limit=4,
    general_params={'use_algos': [['lgb', 'cb', 'linear']]},
    reader_params={'cv': 5, 'random_state': 42}
)

# Обучение
oof_pred = automl.fit_predict(train_features, roles={'target': 'SalePrice'})

# Оценка на CV
mae_cv = mean_absolute_error(train_target, oof_pred.data[:, 0])
print(f"MAE на cross-validation: {mae_cv}")

# Сохранение отчета в файл (поскольку в PyCharm может не работать интерактивно)
with open('automl_report.html', 'w') as f:
    f.write(automl.create_model_str_desc())

/home/stranger/PycharmProjects/ml-advanced/.venv/lib/python3.11/site-packages/lightautoml/transformers/categorical.py:1062: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  cnts = concat([cnts, Series([cnts.shape[0] + 1], index=[np.nan])])


MAE на cross-validation: 15712.3173828125


In [28]:
# 5. Предсказания на тесте
test_pred = automl.predict(test_features)
test_pred_df = pd.DataFrame({'Id': test_df['Id'], 'SalePrice': test_pred.data[:, 0]})
test_pred_df.to_csv('submission.csv', index=False)

# Приближенная оценка MAE
# Для демонстрации, разделим train на train/val
X_train, X_val, y_train, y_val = train_test_split(train_features.drop('SalePrice', axis=1), train_target, test_size=0.2, random_state=42)
automl_temp = TabularAutoML(task=task, timeout=300, cpu_limit=4)
automl_temp.fit_predict(pd.concat([X_train, y_train], axis=1), roles={'target': 'SalePrice'})
val_pred = automl_temp.predict(X_val)
mae_val = mean_absolute_error(y_val, val_pred.data[:, 0])

# Вывод MAE
print(f"MAE CV: {mae_cv}, MAE Val: {mae_val}")

/home/stranger/PycharmProjects/ml-advanced/.venv/lib/python3.11/site-packages/lightautoml/transformers/categorical.py:1062: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  cnts = concat([cnts, Series([cnts.shape[0] + 1], index=[np.nan])])


MAE CV: 15712.3173828125, MAE Val: 16139.955078125


In [29]:
# 6. Улучшенный AutoML с дополнительными настройками
utilized_automl = TabularUtilizedAutoML(
    task=task,
    timeout=1200,  # 20 минут
    cpu_limit=4,
    general_params={
        'use_algos': [['lgb_tuned', 'cb_tuned', 'linear']],
        'tuning_params': {'max_tuning_iter': 20}
    },
    reader_params={'cv': 5, 'random_state': 42},
    tuning_params={'fit_on_holdout': True}
)

# Обучение улучшенной модели
oof_pred_improved = utilized_automl.fit_predict(train_features, roles={'target': 'SalePrice'})

# Оценка
mae_cv_improved = mean_absolute_error(train_target, oof_pred_improved.data[:, 0])
print(f"Улучшенный MAE на CV: {mae_cv_improved}")

# Предсказания и сабмит
test_pred_improved = utilized_automl.predict(test_features)
test_pred_improved_df = pd.DataFrame({'Id': test_df['Id'], 'SalePrice': test_pred_improved.data[:, 0]})
test_pred_improved_df.to_csv('submission_improved.csv', index=False)

# Запись отчета
with open('improved_automl_report.html', 'w') as f:
    f.write(utilized_automl.create_model_str_desc())

print(f"MAE CV Improved: {mae_cv_improved}")

/home/stranger/PycharmProjects/ml-advanced/.venv/lib/python3.11/site-packages/lightautoml/transformers/categorical.py:1062: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  cnts = concat([cnts, Series([cnts.shape[0] + 1], index=[np.nan])])
/home/stranger/PycharmProjects/ml-advanced/.venv/lib/python3.11/site-packages/lightautoml/transformers/categorical.py:1062: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  cnts = concat([cnts, Series([cnts.shape[0] + 1], index=[np.nan])])
/home/stranger/PycharmProjects/ml-advanced/.venv/lib/python3.11/site-packages/lightaut

Улучшенный MAE на CV: 14166.021484375
MAE CV Improved: 14166.021484375


In [31]:
# 7. Пример дополнительного этапа: настройка roles для feature engineering
roles = {
    'target': 'SalePrice',
    'drop': ['Id'],  # Удалить неинформативные фичи
}

# Новый AutoML с roles
final_automl = TabularUtilizedAutoML(
    task=task,
    timeout=1800,  # 30 минут
    cpu_limit=4,
    general_params={
        'use_algos': [['lgb_tuned', 'cb_tuned', 'linear', 'mlp']],
        'tuning_params': {'max_tuning_iter': 30}
    },
    reader_params={'cv': 5, 'random_state': 42},
    tuning_params={'fit_on_holdout': True}
)

oof_pred_final = final_automl.fit_predict(train_features, roles=roles)
mae_cv_final = mean_absolute_error(train_target, oof_pred_final.data[:, 0])
print(f"Финальный MAE на CV: {mae_cv_final}")

if mae_cv_final <= 15324.25:
    test_pred_final = final_automl.predict(test_features)
    test_pred_final_df = pd.DataFrame({'Id': test_df['Id'], 'SalePrice': test_pred_final.data[:, 0]})
    test_pred_final_df.to_csv('submission_final.csv', index=False)
    print("Метрика достигнута!")
else:
    print("Нужно еще улучшить модель.")

# Запись финального отчета
with open('final_automl_report.html', 'w') as f:
    f.write(final_automl.create_model_str_desc())

print(f"MAE CV Final: {mae_cv_final}")

/home/stranger/PycharmProjects/ml-advanced/.venv/lib/python3.11/site-packages/lightautoml/transformers/categorical.py:1062: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  cnts = concat([cnts, Series([cnts.shape[0] + 1], index=[np.nan])])
/home/stranger/PycharmProjects/ml-advanced/.venv/lib/python3.11/site-packages/lightautoml/transformers/categorical.py:1062: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  cnts = concat([cnts, Series([cnts.shape[0] + 1], index=[np.nan])])
/home/stranger/PycharmProjects/ml-advanced/.venv/lib/python3.11/site-packages/lightaut

Финальный MAE на CV: 14174.92578125
Метрика достигнута!
MAE CV Final: 14174.92578125


In [36]:
#### Общий вывод: на обоих циклах обучения с помощью TabularUtilizedAutoML удалось достичь приемлемого результата, можно и дальше эксперементировать с их настройками для улучшения MAE